# ⚠️ DEPRECATED: Automated SDK Import

## This notebook is deprecated due to timeout issues with SDK import.

### Known Issues:

- ❌ 5-minute timeout per dashboard
- ❌ Fails on large dashboards
- ❌ No retry logic
- ❌ Synchronous/blocking operations

### Alternative Approaches (Recommended):

**Option 1: Bundle Workflow (Recommended)**
1. Run `Bundle/Bundle_01_Export_and_Transform.ipynb`
2. Run `Bundle/Bundle_02_Generate_and_Deploy.ipynb`

**Option 2: Manual Import + Permissions**
1. Run `02_Export_and_Transform.ipynb`
2. Import dashboards manually (UI or Bundle)
3. Run `03A_Apply_Permissions.ipynb`

### This file is kept for reference only.

---


# Lakeview Dashboard Migration - Part 3: Import & Migrate

## Overview

This is the third and final notebook for migrating Databricks Lakeview dashboards.

### Three-Notebook Workflow

1. **Notebook 1 (Setup)**: ✅ COMPLETED - Authentication and volume configured
2. **Notebook 2 (Export & Transform)**: ✅ COMPLETED - Dashboards exported and transformed
3. **This Notebook (Import & Migrate)**: Import to target workspace and restore permissions

## What This Notebook Does

### Choose Your Workflow:

#### Option A: Manual Import (Cells 5-7)
- 📥 View manual import instructions
- 🖱️  Import dashboards via Databricks UI
- 🔐 Apply ACLs to manually imported dashboards
- 📊 Generate manual workflow report

#### Option B: Automated Import (Cells 8-9)
- 🤖 Programmatically import all dashboards
- 🔐 Automatically restore permissions
- 📊 Generate automated workflow report

## Prerequisites

✅ Complete Notebook 1: Setup & Configuration  
✅ Complete Notebook 2: Export & Transform

---

In [ ]:
# Install/upgrade required packages
%pip install -U databricks-sdk pandas --quiet
dbutils.library.restartPython()

## Cell 1: Import Configuration

**Purpose:** Import configuration from previous notebooks.

**Instructions:** Copy configuration values from Notebook 1.

In [ ]:
# Import libraries
import json
import csv
import re
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from pathlib import Path
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

# ============================================================================
# AUTHENTICATION CONFIGURATION (from Notebook 1)
# ============================================================================

AUTH_METHOD = "oauth"  # Options: "oauth", "service_principal", "pat"

SOURCE_WORKSPACE_URL = "https://your-source-workspace.cloud.databricks.com"
TARGET_WORKSPACE_URL = "https://your-target-workspace.cloud.databricks.com"

# Load credentials
if AUTH_METHOD == "service_principal":
    SOURCE_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="source-sp-client-id")
    SOURCE_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="source-sp-secret")
    SOURCE_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="source-sp-tenant")
    TARGET_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="target-sp-client-id")
    TARGET_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="target-sp-secret")
    TARGET_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="target-sp-tenant")
elif AUTH_METHOD == "pat":
    SOURCE_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="source-token")
    TARGET_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="target-token")

# ============================================================================
# VOLUME CONFIGURATION (from Notebook 1)
# ============================================================================

VOLUME_BASE_PATH = "/Volumes/migration_catalog/migration_schema/migration_volume/dashboard_migration"
EXPORT_PATH = f"{VOLUME_BASE_PATH}/exported"
TRANSFORMED_PATH = f"{VOLUME_BASE_PATH}/transformed"
LOGS_PATH = f"{VOLUME_BASE_PATH}/logs"

# ============================================================================
# TARGET CONFIGURATION
# ============================================================================

TARGET_FOLDER_PATH = "/Workspace/Shared/Migrated_Dashboards"

# REQUIRED: Warehouse ID for running dashboards in target workspace
TARGET_WAREHOUSE_ID = "your-warehouse-id"  # UPDATE THIS - Get from SQL Warehouse settings

# ============================================================================
# OPTIONS
# ============================================================================

SKIP_PERMISSIONS = False  # Set True to skip permissions restoration
DRY_RUN = False          # Set True to test without importing

print("✅ Configuration loaded")
print(f"   Auth: {AUTH_METHOD}")
print(f"   Target: {TARGET_WORKSPACE_URL}")
print(f"   Target folder: {TARGET_FOLDER_PATH}")
print(f"   Dry run: {DRY_RUN}")

## Cell 2: Helper Functions - Authentication & Volume I/O

**Purpose:** Define helper functions for authentication and volume operations.

In [ ]:
def create_workspace_client(workspace_url: str, is_source: bool = True) -> WorkspaceClient:
    """Create WorkspaceClient with configured authentication."""
    if AUTH_METHOD == "service_principal":
        if is_source:
            return WorkspaceClient(
                host=workspace_url,
                client_id=SOURCE_SP_CLIENT_ID,
                client_secret=SOURCE_SP_CLIENT_SECRET,
                azure_tenant_id=SOURCE_SP_TENANT_ID
            )
        else:
            return WorkspaceClient(
                host=workspace_url,
                client_id=TARGET_SP_CLIENT_ID,
                client_secret=TARGET_SP_CLIENT_SECRET,
                azure_tenant_id=TARGET_SP_TENANT_ID
            )
    elif AUTH_METHOD == "oauth":
        return WorkspaceClient(host=workspace_url)
    elif AUTH_METHOD == "pat":
        token = SOURCE_PAT_TOKEN if is_source else TARGET_PAT_TOKEN
        return WorkspaceClient(host=workspace_url, token=token)
    else:
        raise ValueError(f"Unknown auth method: {AUTH_METHOD}")

def read_volume_file(volume_path: str) -> str:
    """Read file from volume (serverless-compatible)."""
    content = dbutils.fs.head(volume_path, 10485760)  # Read up to 10MB
    return content

def write_volume_file(volume_path: str, content: str) -> None:
    """Write file to volume (serverless-compatible)."""
    dbutils.fs.put(volume_path, content, overwrite=True)

def list_volume_files(volume_path: str, pattern: str = "*") -> List[str]:
    """List files in volume directory."""
    try:
        files = dbutils.fs.ls(volume_path)
        matched = []
        for file_info in files:
            file_path = file_info.path.replace("dbfs:", "")
            if pattern == "*" or file_path.endswith(pattern.replace("*", "")):
                matched.append(file_path)
        return matched
    except:
        return []

print("✅ Helper functions loaded")

## Cell 3: Import & Permissions Helper Functions

**Purpose:** Functions for importing dashboards and restoring permissions.

In [ ]:
def import_dashboard_from_lvdash(
    client: WorkspaceClient,
    lvdash_content: str,
    dashboard_name: str,
    target_folder_path: str,
    warehouse_id: str,
    dry_run: bool = False
) -> Dict:
    """Import dashboard using Lakeview API."""
    if dry_run:
        return {
            "status": "dry_run",
            "dashboard_id": None,
            "message": "Dry run - no import performed"
        }
    
    try:
        # Parse dashboard content to get display name
        dashboard_data = json.loads(lvdash_content)
        display_name = dashboard_data.get("display_name", dashboard_name)
        
        # Create dashboard using lakeview API with proper structure
        # The SDK expects specific named parameters
        from databricks.sdk.service.dashboards import Dashboard
        
        dashboard = Dashboard(
            display_name=display_name,
            parent_path=target_folder_path,
            warehouse_id=warehouse_id,
            serialized_dashboard=lvdash_content
        )
        
        created_dashboard = client.lakeview.create(dashboard=dashboard)
        
        return {
            "status": "success",
            "dashboard_id": created_dashboard.dashboard_id,
            "dashboard_name": created_dashboard.display_name
        }
    except Exception as e:
        # Catch all exceptions and return error details
        error_msg = str(e)
        if "not found" in error_msg.lower():
            error_type = "Resource not found"
        elif "permission" in error_msg.lower() or "denied" in error_msg.lower():
            error_type = "Permission denied"
        else:
            error_type = "API error"
        
        return {
            "status": "failed",
            "dashboard_id": None,
            "error": f"{error_type}: {error_msg}"
        }


## Cell 4: List Transformed Dashboards

**Purpose:** Display available transformed dashboards ready for import.

**Choose your workflow below:**
- **Manual Import**: Go to Cell 5
- **Automated Import**: Go to Cell 8

In [ ]:
print("=" * 80)
print("📂 AVAILABLE TRANSFORMED DASHBOARDS")
print("=" * 80)

# Get transformed dashboard files
transformed_files = list_volume_files(TRANSFORMED_PATH, "*.lvdash.json")

if not transformed_files:
    print("\n⚠️  No transformed dashboards found!")
    print("   Make sure Notebook 2 (Export & Transform) completed successfully")
    print(f"   Expected location: {TRANSFORMED_PATH}")
else:
    print(f"\nFound {len(transformed_files)} transformed dashboard(s):\n")
    
    # Parse and display dashboard info
    dashboard_info = []
    for file_path in transformed_files:
        filename = Path(file_path).name
        
        # Extract dashboard ID and name from filename
        match = re.match(r'dashboard_([^_]+)_(.+)\.lvdash\.json', filename)
        if match:
            dash_id = match.group(1)
            dash_name = match.group(2).replace('_', ' ')
            
            dashboard_info.append({
                "id": dash_id,
                "name": dash_name,
                "file": file_path
            })
    
    # Display as table
    if dashboard_info:
        df = pd.DataFrame(dashboard_info)
        display(df)
    
    print("\n" + "=" * 80)
    print("CHOOSE YOUR IMPORT WORKFLOW")
    print("=" * 80)
    
    print("\n📋 Option A: Manual Import (CELLS 5-7)")
    print("   - View import instructions")
    print("   - Import dashboards via Databricks UI")
    print("   - Apply ACLs using this notebook")
    print("   - Good for: Review before import, selective migration")
    
    print("\n🤖 Option B: Automated Import (CELLS 8-9)")
    print("   - Programmatic import of all dashboards")
    print("   - Automatic ACL restoration")
    print("   - Good for: Batch processing, repeated migrations")
    
    print("\n" + "=" * 80)
    print("👉 Choose one workflow and run the corresponding cells")
    print("=" * 80)

---

# MANUAL IMPORT WORKFLOW

## Cells 5-7: Manual Import Process

**Use this workflow if you want to:**
- Review dashboards before importing
- Import selectively via Databricks UI
- Have more control over the import process

---

## Cell 5: Manual Import Instructions

**Purpose:** Display instructions for manually importing dashboards via Databricks UI.

**Workflow:** MANUAL IMPORT - Step 1

In [ ]:
print("=" * 80)
print("🔧 MANUAL IMPORT WORKFLOW - Step 1: Import Instructions")
print("=" * 80)

print(f"\n📁 Transformed dashboards are ready in:")
print(f"   {TRANSFORMED_PATH}")

print("\n📋 HOW TO MANUALLY IMPORT:")
print("\n1. Access the volume folder in Databricks:")
print(f"   - Navigate to: Data → {TRANSFORMED_PATH}")
print("   - Or use Catalog Explorer")

print("\n2. Download .lvdash.json files:")
print("   - Download the dashboards you want to import")
print("   - Save them to your local machine")

print("\n3. Import via Databricks UI:")
print(f"   - Go to target workspace: {TARGET_WORKSPACE_URL}")
print("   - Navigate to: Workspace → Import")
print("   - Select and upload the .lvdash.json file(s)")
print(f"   - Choose destination folder: {TARGET_FOLDER_PATH}")
print("   - Click Import")

print("\n4. Note the imported dashboard paths:")
print("   - After import, note the full workspace path of each dashboard")
print("   - You'll need these paths for Cell 6 (ACL application)")

print("\n" + "=" * 80)
print("AVAILABLE DASHBOARDS TO IMPORT")
print("=" * 80)

# List available files
transformed_files = list_volume_files(TRANSFORMED_PATH, "*.lvdash.json")
print(f"\nTotal: {len(transformed_files)} dashboard(s)\n")

for i, file_path in enumerate(transformed_files, 1):
    filename = Path(file_path).name
    match = re.match(r'dashboard_([^_]+)_(.+)\.lvdash\.json', filename)
    if match:
        dash_id = match.group(1)
        dash_name = match.group(2).replace('_', ' ')
        print(f"   {i}. ID: {dash_id} | Name: {dash_name}")
        print(f"      File: {filename}")
    else:
        print(f"   {i}. {filename}")

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("\n1. Import the dashboards using the instructions above")
print("2. After importing, configure MANUAL_IMPORT_MAPPING in Cell 6")
print("3. Run Cell 6 to apply ACLs to the imported dashboards")
print("4. Run Cell 7 to generate the migration report")

## Cell 6: Apply ACLs to Manual Imports

**Purpose:** Apply permissions to dashboards that were manually imported.

**Workflow:** MANUAL IMPORT - Step 2

**Instructions:**
1. After manually importing dashboards (Cell 5), update MANUAL_IMPORT_MAPPING below
2. Map: `"old_dashboard_id": "new_dashboard_path_in_target"`
3. Run this cell to apply ACLs

In [ ]:
# ============================================================================
# CONFIGURATION: Manual Import Mapping
# ============================================================================
# Update this mapping after manually importing dashboards
# Format: "old_dashboard_id": "new_dashboard_path_in_target_workspace"

MANUAL_IMPORT_MAPPING = {
    # Example:
    # "abc123": "/Workspace/Shared/Migrated_Dashboards/Sales Dashboard",
    # "def456": "/Workspace/Shared/Migrated_Dashboards/Marketing KPIs",
}

# ============================================================================
# APPLY ACLS
# ============================================================================

if not MANUAL_IMPORT_MAPPING:
    print("=" * 80)
    print("⚠️  MANUAL_IMPORT_MAPPING IS EMPTY")
    print("=" * 80)
    print("\nPlease configure the mapping above before running this cell.")
    print("\nFormat:")
    print("   MANUAL_IMPORT_MAPPING = {")
    print("       'old_dashboard_id': '/Workspace/path/to/imported/dashboard',")
    print("   }")
    print("\n👉 After configuring, re-run this cell to apply ACLs")
else:
    print("=" * 80)
    print("🔐 MANUAL IMPORT WORKFLOW - Step 2: Apply ACLs")
    print("=" * 80)
    
    # Create target client
    target_client = create_workspace_client(TARGET_WORKSPACE_URL, is_source=False)
    print(f"\n✅ Connected to target workspace")
    
    acl_results = []
    
    print(f"\nProcessing {len(MANUAL_IMPORT_MAPPING)} dashboard(s)...\n")
    
    for idx, (old_id, new_path) in enumerate(MANUAL_IMPORT_MAPPING.items(), 1):
        print(f"[{idx}/{len(MANUAL_IMPORT_MAPPING)}] Processing: {new_path}")
        
        try:
            # Find permissions file
            perms_files = [f for f in list_volume_files(EXPORT_PATH, "*.json") 
                          if old_id in f and "permissions" in f]
            
            if not perms_files:
                print(f"   ⚠️  No permissions file found for ID: {old_id}")
                acl_results.append({
                    "dashboard_id": old_id,
                    "target_path": new_path,
                    "status": "no_permissions_file",
                    "applied": 0,
                    "skipped": 0
                })
                continue
            
            # Read permissions
            perms_content = read_volume_file(perms_files[0])
            permissions = json.loads(perms_content)
            
            # Apply permissions
            perms_result = apply_dashboard_permissions(
                client=target_client,
                dashboard_path=new_path,
                permissions=permissions,
                skip_permissions=SKIP_PERMISSIONS
            )
            
            print(f"   ✅ Applied: {perms_result['applied']}, Skipped: {perms_result['skipped']}")
            
            acl_results.append({
                "dashboard_id": old_id,
                "target_path": new_path,
                "status": "success",
                "applied": perms_result["applied"],
                "skipped": perms_result["skipped"]
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            acl_results.append({
                "dashboard_id": old_id,
                "target_path": new_path,
                "status": "error",
                "error": str(e),
                "applied": 0,
                "skipped": 0
            })
    
    # Summary
    print("\n" + "=" * 80)
    print("📊 ACL APPLICATION SUMMARY")
    print("=" * 80)
    
    successful = len([r for r in acl_results if r["status"] == "success"])
    total_applied = sum(r.get("applied", 0) for r in acl_results)
    total_skipped = sum(r.get("skipped", 0) for r in acl_results)
    
    print(f"\n✅ Dashboards processed: {successful}/{len(acl_results)}")
    print(f"🔐 Total permissions applied: {total_applied}")
    print(f"⊘ Total permissions skipped: {total_skipped}")
    
    # Store for report
    manual_acl_results = acl_results
    
    print("\n👉 Next: Run Cell 7 to generate the migration report")

## Cell 7: Manual Workflow Report

**Purpose:** Generate migration report for manual workflow.

**Workflow:** MANUAL IMPORT - Step 3

In [ ]:
print("=" * 80)
print("📊 MANUAL WORKFLOW MIGRATION REPORT")
print("=" * 80)

if 'manual_acl_results' not in locals():
    print("\n⚠️  No ACL results found.")
    print("   Run Cell 6 first to apply ACLs")
else:
    # Create report DataFrame
    report_data = []
    
    for result in manual_acl_results:
        report_data.append({
            "Dashboard ID": result["dashboard_id"],
            "Target Path": result["target_path"],
            "Status": result["status"],
            "Permissions Applied": result.get("applied", 0),
            "Permissions Skipped": result.get("skipped", 0),
            "Error": result.get("error", "")
        })
    
    df_report = pd.DataFrame(report_data)
    
    print("\n📋 DASHBOARD MIGRATION RESULTS:\n")
    display(df_report)
    
    # Statistics
    print("\n" + "=" * 80)
    print("📈 STATISTICS")
    print("=" * 80)
    
    total = len(report_data)
    successful = len([r for r in manual_acl_results if r["status"] == "success"])
    total_applied = sum(r.get("applied", 0) for r in manual_acl_results)
    total_skipped = sum(r.get("skipped", 0) for r in manual_acl_results)
    
    print(f"\nTotal dashboards: {total}")
    print(f"Successfully processed: {successful} ({(successful/total*100):.1f}%)")
    print(f"Permissions applied: {total_applied}")
    print(f"Permissions skipped: {total_skipped}")
    
    # Save report to volume
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = f"{LOGS_PATH}/manual_migration_report_{timestamp}.json"
    
    report_json = {
        "timestamp": timestamp,
        "workflow": "manual",
        "total": total,
        "successful": successful,
        "permissions_applied": total_applied,
        "permissions_skipped": total_skipped,
        "dashboards": manual_acl_results
    }
    
    write_volume_file(report_file, json.dumps(report_json, indent=2))
    print(f"\n💾 Report saved to: {report_file}")
    
    print("\n" + "=" * 80)
    print("✅ MANUAL WORKFLOW COMPLETE!")
    print("=" * 80)
    print("\nYour dashboards have been migrated to the target workspace.")
    print(f"Target workspace: {TARGET_WORKSPACE_URL}")
    print(f"\nDashboard location(s):")
    for result in manual_acl_results[:5]:  # Show first 5
        print(f"   - {result['target_path']}")
    if len(manual_acl_results) > 5:
        print(f"   ... and {len(manual_acl_results) - 5} more")

---

# AUTOMATED IMPORT WORKFLOW

## Cells 8-9: Automated Import Process

**Use this workflow if you want to:**
- Import all dashboards programmatically
- Automate the entire migration process
- Batch process multiple dashboards

**Note:** Skip Cells 5-7 if using automated workflow.

---

## Cell 8: Automated Import

**Purpose:** Automatically import all transformed dashboards to target workspace.

**Workflow:** AUTOMATED IMPORT

In [ ]:
print("=" * 80)
print("🤖 AUTOMATED IMPORT WORKFLOW")
print("=" * 80)

if DRY_RUN:
    print("\n⚠️  DRY RUN MODE ENABLED")
    print("   No actual imports will be performed")
    print("   Set DRY_RUN = False to perform real imports\n")

# Create target client
target_client = create_workspace_client(TARGET_WORKSPACE_URL, is_source=False)
print(f"\n✅ Connected to target workspace")

# Get transformed dashboards
transformed_files = list_volume_files(TRANSFORMED_PATH, "*.lvdash.json")
print(f"\n📂 Found {len(transformed_files)} transformed dashboard(s)")

if not transformed_files:
    print("\n⚠️  No dashboards to import!")
else:
    import_results = []
    
    print("\n📥 Importing dashboards...\n")
    
    for idx, dashboard_file in enumerate(transformed_files, 1):
        filename = Path(dashboard_file).name
        
        # Extract dashboard info
        match = re.match(r'dashboard_([^_]+)_(.+)\.lvdash\.json', filename)
        if match:
            dash_id = match.group(1)
            dash_name = match.group(2).replace('_', ' ')
        else:
            dash_id = filename
            dash_name = filename
        
        print(f"[{idx}/{len(transformed_files)}] Importing: {dash_name}")
        
        try:
            # Read dashboard content
            lvdash_content = read_volume_file(dashboard_file)
            
            # Import dashboard
            import_result = import_dashboard_from_lvdash(
                client=target_client,
                lvdash_content=lvdash_content,
                dashboard_name=dash_name,
                target_folder_path=TARGET_FOLDER_PATH,
                warehouse_id=TARGET_WAREHOUSE_ID,
                dry_run=DRY_RUN
            )
            
            target_path = f"{TARGET_FOLDER_PATH}/{dash_name}"
            
            if import_result["status"] == "success":
                print(f"   ✅ Imported to: {target_path}")
            elif import_result["status"] == "dry_run":
                print(f"   🔍 Dry run - would import to: {target_path}")
            else:
                print(f"   ❌ Failed: {import_result.get('error', 'Unknown error')}")
            
            # Get permissions file
            perms_files = [f for f in list_volume_files(EXPORT_PATH, "*.json") 
                          if dash_id in f and "permissions" in f]
            
            # Apply permissions if not dry run and not skipped
            if not DRY_RUN and not SKIP_PERMISSIONS and import_result["status"] == "success":
                if perms_files:
                    perms_content = read_volume_file(perms_files[0])
                    permissions = json.loads(perms_content)
                    
                    perms_result = apply_dashboard_permissions(
                        client=target_client,
                        dashboard_path=target_path,
                        permissions=permissions,
                        skip_permissions=SKIP_PERMISSIONS
                    )
                    
                    print(f"   🔐 Permissions - Applied: {perms_result['applied']}, Skipped: {perms_result['skipped']}")
                    import_result["permissions"] = perms_result
                else:
                    print(f"   ⚠️  No permissions file found")
                    import_result["permissions"] = {"applied": 0, "skipped": 0}
            else:
                import_result["permissions"] = {"applied": 0, "skipped": 0}
            
            import_results.append({
                "dashboard_id": dash_id,
                "dashboard_name": dash_name,
                "source_file": dashboard_file,
                "target_path": target_path,
                "import_status": import_result["status"],
                "permissions_applied": import_result.get("permissions", {}).get("applied", 0),
                "permissions_skipped": import_result.get("permissions", {}).get("skipped", 0),
                "error": import_result.get("error", "")
            })
            
        except Exception as e:
            print(f"   ❌ Unexpected error: {e}")
            import_results.append({
                "dashboard_id": dash_id,
                "dashboard_name": dash_name,
                "import_status": "failed",
                "error": str(e),
                "permissions_applied": 0,
                "permissions_skipped": 0
            })
    
    # Store results for Cell 9
    automated_import_results = import_results
    
    # Summary
    print("\n" + "=" * 80)
    print("📊 IMPORT SUMMARY")
    print("=" * 80)
    
    successful = len([r for r in import_results if r["import_status"] == "success"])
    failed = len([r for r in import_results if r["import_status"] == "failed"])
    
    print(f"\n✅ Successful: {successful}")
    print(f"❌ Failed: {failed}")
    
    if not DRY_RUN:
        total_applied = sum(r.get("permissions_applied", 0) for r in import_results)
        total_skipped = sum(r.get("permissions_skipped", 0) for r in import_results)
        print(f"🔐 Permissions applied: {total_applied}")
        print(f"⊘ Permissions skipped: {total_skipped}")
    
    print("\n👉 Next: Run Cell 9 to generate the migration report")

## Cell 9: Automated Workflow Report

**Purpose:** Generate comprehensive migration report for automated workflow.

**Workflow:** AUTOMATED IMPORT - Final Report

In [ ]:
print("=" * 80)
print("📊 AUTOMATED WORKFLOW MIGRATION REPORT")
print("=" * 80)

if 'automated_import_results' not in locals():
    print("\n⚠️  No import results found.")
    print("   Run Cell 8 first to import dashboards")
else:
    # Create report DataFrame
    df_report = pd.DataFrame(automated_import_results)
    
    print("\n📋 DASHBOARD MIGRATION RESULTS:\n")
    display(df_report)
    
    # Statistics
    print("\n" + "=" * 80)
    print("📈 MIGRATION STATISTICS")
    print("=" * 80)
    
    total = len(automated_import_results)
    successful = len([r for r in automated_import_results if r["import_status"] == "success"])
    failed = len([r for r in automated_import_results if r["import_status"] == "failed"])
    dry_run_count = len([r for r in automated_import_results if r["import_status"] == "dry_run"])
    
    print(f"\nTotal dashboards: {total}")
    print(f"Successfully imported: {successful} ({(successful/total*100 if total > 0 else 0):.1f}%)")
    print(f"Failed: {failed}")
    
    if dry_run_count > 0:
        print(f"Dry run: {dry_run_count}")
    
    if not DRY_RUN:
        total_applied = sum(r.get("permissions_applied", 0) for r in automated_import_results)
        total_skipped = sum(r.get("permissions_skipped", 0) for r in automated_import_results)
        print(f"\nPermissions applied: {total_applied}")
        print(f"Permissions skipped: {total_skipped}")
    
    # Save report to volume
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = f"{LOGS_PATH}/automated_migration_report_{timestamp}.json"
    
    report_json = {
        "timestamp": timestamp,
        "workflow": "automated",
        "dry_run": DRY_RUN,
        "total": total,
        "successful": successful,
        "failed": failed,
        "permissions_applied": sum(r.get("permissions_applied", 0) for r in automated_import_results),
        "permissions_skipped": sum(r.get("permissions_skipped", 0) for r in automated_import_results),
        "dashboards": automated_import_results
    }
    
    write_volume_file(report_file, json.dumps(report_json, indent=2))
    print(f"\n💾 Report saved to: {report_file}")
    
    # Final message
    print("\n" + "=" * 80)
    if DRY_RUN:
        print("🔍 DRY RUN COMPLETE!")
        print("=" * 80)
        print("\nThis was a dry run. No actual imports were performed.")
        print("\nTo perform the real migration:")
        print("1. Set DRY_RUN = False in Cell 1")
        print("2. Re-run Cell 8 and Cell 9")
    else:
        print("✅ AUTOMATED MIGRATION COMPLETE!")
        print("=" * 80)
        print(f"\n🎉 Successfully migrated {successful} dashboard(s) to target workspace!")
        print(f"\nTarget workspace: {TARGET_WORKSPACE_URL}")
        print(f"Dashboard folder: {TARGET_FOLDER_PATH}")
        
        if failed > 0:
            print(f"\n⚠️  {failed} dashboard(s) failed to import. Check the report above for details.")
        
        print("\n📋 Next steps:")
        print("1. Verify dashboards in target workspace")
        print("2. Test dashboard functionality")
        print("3. Verify permissions are correct")
        print("4. Review migration report for any issues")

## Migration Complete! 🎉

### What You've Accomplished

#### Across All Three Notebooks:

1. **Notebook 1 (Setup)**:
   - ✅ Configured authentication
   - ✅ Created volume structure
   - ✅ Prepared CSV mappings

2. **Notebook 2 (Export & Transform)**:
   - ✅ Exported dashboards from source
   - ✅ Captured permissions
   - ✅ Applied catalog/schema/table transformations

3. **This Notebook (Import & Migrate)**:
   - ✅ Imported dashboards to target workspace
   - ✅ Restored permissions
   - ✅ Generated migration report

### Files Created

- **Volume artifacts**: Exported, transformed dashboards, and permissions
- **Migration reports**: Detailed JSON logs in `{LOGS_PATH}/`
- **Target dashboards**: Imported and ready to use

### Recommended Next Steps

1. **Verify Dashboards**:
   - Open each dashboard in target workspace
   - Verify queries execute successfully
   - Check data displays correctly

2. **Verify Permissions**:
   - Check dashboard sharing settings
   - Confirm users/groups have appropriate access

3. **Test End-to-End**:
   - Run dashboard refreshes
   - Test filters and parameters
   - Verify drill-downs work

4. **Clean Up** (optional):
   - Archive exported/transformed files
   - Remove source dashboards (if migrating, not copying)

---

**Questions or Issues?** Refer to the comprehensive guide document or migration reports.